# 03 · T9 and T11 — Baseline methods (DiCE, Wachter, FACE)

**Deliverable (plan, section 5):** DiCE, Wachter and FACE behind Mary's common interface (`CounterfactualMethod`).

| method | file | runs on | note |
|---|---|---|---|
| DiCE | `cfqd/methods/dice.py` | LR, RF, NN | `dice-ml` 0.12; `genetic` (main) and `random` (floor) |
| Wachter | `cfqd/methods/wachter.py` | LR, NN | CARLA port (same loss, λ schedule and stop rule); budget in queries |
| FACE | `cfqd/methods/face.py` | LR, RF, NN | extracted from CARLA, model through `predict_proba`; CARLA bugs fixed |
| MOC | — | — | **owner still open** (pending item 1); not implemented here |

Every query goes through `CountingModel` (Wachter charges each gradient step with `CountingModel.charge`).

**Open decision:** the predictions on the training data used by DiCE (KD-tree) and FACE (graph) are served
**uncounted** (`train_predictions: free`), amortized across instances; with `counted`, DiCE would pay 31k queries per
instance on Adult and fail at every budget.

**Fixes made after the first CARLA check (20/11):**
* **DiCE stalled on COMPAS** (success 0.00): `dice-ml` retrieves 1,000 KD-tree neighbours but builds the population
  from the first 100; with the immutables fixed those collapse into a few distinct points and dice-ml fills the gap
  with uniform random draws predicted one row at a time until they are valid — on COMPAS a draw is valid with
  probability 0.035%, so the whole budget was burned there. `_kd_init_fill` keeps dice-ml's own first block, caps the
  random draws at a fifth of the remaining budget, then uses the other retrieved neighbours.
* **FACE** now checks up to 500 candidates (was 50) after projecting them onto the constraints.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

In [2]:
from cfqd import CountingModel, datasets as D, models as M
from cfqd.methods import Dice, Face, MapElites, Wachter
from cfqd.metrics import constraint_violations, guidotti_distance
from cfqd.experiment import load_summaries, run_grid

ds = D.load_dataset("german_credit", seed=0)
model = M.get_model(ds, "nn")
cm = CountingModel(model)
pos = int(np.flatnonzero(cm.predict_uncounted(ds.X_test) == 0)[0])
x = ds.X_test[pos]
print("instance (row", int(ds.idx_test[pos]), ") P(favorable) =", round(float(cm.predict_proba_uncounted(x)[0, 1]), 3))

instance (row 13 ) P(favorable) = 0.387


## 1. One instance, every method (German Credit, neural network, budget 2,000)

In [3]:
methods = {
    "dice_genetic": Dice(ds.X_train, ds.y_train, ds.spec, method="genetic", total_cfs=5),
    "wachter": Wachter(),
    "face_knn": Face(ds.X_train),
    "map_elites": MapElites(),
}
res = {k: m.explain(x, cm, ds.spec, desired=1, budget=2000, seed=0) for k, m in methods.items()}
pd.DataFrame([dict(method=k, cfs=len(r.cfs), valid=int((cm.predict_uncounted(r.cfs) == 1).sum()) if len(r.cfs) else 0,
                   queries=r.n_queries, seconds=round(r.seconds, 2),
                   violations=sum(constraint_violations(x, c, ds.spec) for c in r.cfs)) for k, r in res.items()])

,method,cfs,valid,queries,seconds,violations
0,dice_genetic,5,5,337,1.36,0
1,wachter,1,1,16,2.43,0
2,face_knn,1,1,1,3.56,0
3,map_elites,64,64,2000,0.38,0


In [4]:
def changes(x, cf, spec):
    """Changed features: original -> counterfactual."""
    return {spec.names[j]: f"{x[j]:g} -> {cf[j]:.4g}" for j in np.flatnonzero(np.abs(x - cf) > 1e-9)}

rows = []
for k, r in res.items():
    cfs = r.cfs[np.argsort([guidotti_distance(x, c, ds.spec) for c in r.cfs])][:3] if len(r.cfs) else []
    for c in cfs:
        rows.append(dict(method=k, distance=round(guidotti_distance(x, c, ds.spec), 3), changes=changes(x, c, ds.spec)))
pd.DataFrame(rows)

,method,distance,changes
0,dice_genetic,0.373,"{'duration': '48 -> 36', 'purpose': '10 -> 0', 'amount': '7582 -> 8086', 'em..."
1,dice_genetic,0.684,"{'status': '1 -> 0', 'purpose': '10 -> 0', 'amount': '7582 -> 250', 'savings..."
2,dice_genetic,0.684,"{'purpose': '10 -> 9', 'amount': '7582 -> 250', 'savings': '1 -> 0', 'employ..."
3,wachter,0.299,"{'duration': '48 -> 38.29', 'amount': '7582 -> 4998', 'savings': '1 -> 2', '..."
4,face_knn,0.840,"{'duration': '48 -> 6', 'purpose': '10 -> 0', 'amount': '7582 -> 1209', 'sav..."
5,map_elites,0.216,"{'duration': '48 -> 34.05', 'savings': '1 -> 2', 'property': '3 -> 2'}"
6,map_elites,0.236,"{'duration': '48 -> 25.68', 'savings': '1 -> 2'}"
7,map_elites,0.240,"{'duration': '48 -> 25.16', 'property': '3 -> 2'}"


## 2. Check against the literature (CARLA success rates, plan 5.1)

Adult, GMC and COMPAS × LR and NN, 10 unfavorable instances per configuration, budget of 10,000 queries.
Run through the batch runner itself (`run_grid`), so it lands in `results/carla_check/` and is **resumable**.

Caveats: CARLA uses its own models, and its Wachter **ignores constraints**; ours enforces them. These numbers are a
reference range, not an exact target.

In [5]:
exp_carla = dict(name="carla_check", datasets=["adult", "gmc", "compas_guidotti"], models=["lr", "nn"],
                 methods={"dice_genetic": {"total_cfs": 10}, "wachter": {}, "face_knn": {}},
                 seeds=[0], budgets=[10000], n_instances=10, out="results/carla_check",
                 train_predictions="free", model_params={})
_ = run_grid(exp_carla, jobs=1, verbose=True)

18 configurations, 6 already done, 12 to run, 0 skipped


[1/12] adult__lr__dice_genetic__seed0: 44.02s


[2/12] adult__lr__face_knn__seed0: 0.4s


[3/12] adult__nn__dice_genetic__seed0: 38.39s


[4/12] adult__nn__face_knn__seed0: 0.48s


[5/12] gmc__lr__dice_genetic__seed0: 20.46s


... (3 outputs left out)

[9/12] compas_guidotti__lr__dice_genetic__seed0: 94.26s


[10/12] compas_guidotti__lr__face_knn__seed0: 0.83s


[11/12] compas_guidotti__nn__dice_genetic__seed0: 92.75s


[12/12] compas_guidotti__nn__face_knn__seed0: 0.95s


In [6]:
CARLA = {  # success rates published by CARLA (Pawelczyk et al., 2021)
    ("wachter", "adult", "nn"): 0.50, ("wachter", "adult", "lr"): 1.00, ("wachter", "gmc", "nn"): 1.00,
    ("wachter", "gmc", "lr"): 1.00, ("wachter", "compas_guidotti", "nn"): 0.66, ("wachter", "compas_guidotti", "lr"): 1.00,
}
for d in ["adult", "gmc", "compas_guidotti"]:
    for m in ["lr", "nn"]:
        CARLA[("dice_genetic", d, m)] = 1.00
        CARLA[("face_knn", d, m)] = 1.00

s = load_summaries(exp_carla)
sr = s.groupby(["method", "dataset", "model"]).agg(success=("success", "mean"), median_queries=("n_queries", "median"),
                                                     violations=("violations", "sum")).reset_index()
sr["carla"] = [CARLA.get((r.method, r.dataset, r.model)) for r in sr.itertuples()]
sr.pivot_table(index=["method", "model"], columns="dataset", values=["success", "carla"]).round(2)

carla                      success                     
dataset            adult compas_guidotti  gmc   adult compas_guidotti  gmc
method       model                                                        
dice_genetic lr      1.0            1.00  1.0     1.0             0.9  0.9
             nn      1.0            1.00  1.0     1.0             1.0  0.5
face_knn     lr      1.0            1.00  1.0     0.8             0.9  0.9
             nn      1.0            1.00  1.0     0.8             1.0  0.4
wachter      lr      1.0            1.00  1.0     1.0             1.0  1.0
             nn      0.5            0.66  1.0     1.0             1.0  0.8

In [7]:
sr

,method,dataset,model,success,median_queries,violations,carla
0,dice_genetic,adult,lr,1.0,733.0,0,1.00
1,dice_genetic,adult,nn,1.0,773.0,0,1.00
2,dice_genetic,compas_guidotti,lr,0.9,9987.0,0,1.00
3,dice_genetic,compas_guidotti,nn,1.0,5538.0,0,1.00
4,dice_genetic,gmc,lr,0.9,5286.5,0,1.00
5,dice_genetic,gmc,nn,0.5,9993.5,0,1.00
6,face_knn,adult,lr,0.8,1.0,0,1.00
7,face_knn,adult,nn,0.8,1.0,0,1.00
8,face_knn,compas_guidotti,lr,0.9,2.0,0,1.00
9,face_knn,compas_guidotti,nn,1.0,2.5,0,1.00


**Reading.**
* **DiCE** after the fix: compare with the 1.00 of CARLA. Remaining failures, if any, should be instances that the
  constraints make hard or infeasible (e.g. irreversible past-due counters on GMC); check them against MAP-Elites
  on the same instances before calling it a DiCE limitation.
* **FACE** returns *data points*; with irreversible numeric features (GMC past-due counters, German age) the chosen
  point has to be projected onto the instance's frozen values, which often loses the desired class. This is a real
  limitation of FACE under actionability constraints, not a bug.
* **Wachter on Adult + NN reaches 1.00 vs. 0.50 in CARLA.** The plan flags this ("we changed the stop rule or the
  λ schedule"); we kept both. The difference comes from the models (CARLA's ANN vs. ours) and from CARLA's 0.5-minute
  timeout, which our query budget replaces. To be investigated with CARLA's architecture before the paper.

## 3. No counterfactual violates the YAML constraints

Checked on every result written so far, recomputed from the CF rows (not trusting the summaries).

In [8]:
total, n_cf = 0, 0
for f in sorted((ROOT / "results").glob("*/runs/*.csv")):
    df = pd.read_csv(f)
    if df.empty:
        continue
    d = D.load_dataset(df["dataset"].iloc[0], seed=int(df["seed"].iloc[0]))
    for (iid,), g in df.groupby(["instance_id"]):
        x0 = d.X[int(iid)]
        for c in g[d.spec.names].to_numpy(float):
            total += constraint_violations(x0, c, d.spec)
            n_cf += 1
print(f"{n_cf} counterfactuals checked, {total} violations")

100535 counterfactuals checked, 0 violations


## 4. Projection onto the strategy × effort grid

The counterfactuals of **any** method are projected onto the same grid (MAP-Elites is the only one that uses it during
the search). This is the yardstick of RQ1.

In [9]:
from cfqd import Descriptor, GridArchive
desc = Descriptor(ds.spec)
proj = []
for k, r in res.items():
    valid = r.cfs[cm.predict_uncounted(r.cfs) == 1] if len(r.cfs) else r.cfs
    a = GridArchive(desc, x)
    a.add_many(valid)
    proj.append(dict(method=k, valid=len(valid), cells=len(a), strategies=a.n_strategies(),
                     coverage=round(a.coverage(), 4), qd_score=round(a.qd_score(), 2)))
print("feasible cells:", desc.n_feasible_cells)
pd.DataFrame(proj)

feasible cells: 400


,method,valid,cells,strategies,coverage,qd_score
0,dice_genetic,5,5,5,0.0125,3.09
1,wachter,1,1,1,0.0025,0.77
2,face_knn,1,1,1,0.0025,0.54
3,map_elites,64,64,34,0.1600,45.60
